# Tema 2 · Clustering jerárquico

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

La **Cooperativa Campesina Valle del Itata** reúne a pequeños y medianos agricultores de varias comunas de Ñuble. Hoy ofrece **la misma asesoría técnica a todos los socios**, y muchos reclaman que no les sirve. La directiva sospecha que hay **distintos tipos de socio**, pero no sabe cuántos ni cuáles son. Tiene una ficha de cada predio y les pide agruparlos.

> **❓ La pregunta del cliente:** ¿Qué tipos de socio tiene la cooperativa, cuántos son y cómo debería adaptar sus servicios a cada uno?

## 🗂️ Los datos: `t2_predios.csv` (66 predios con su superficie, sus cultivos, sus trabajadores y su distancia a Chillán)

| Columna | Qué significa |
|---|---|
| `predio` | código del predio |
| `comuna` | comuna donde está el predio |
| `hectareas` | superficie del predio, en hectáreas |
| `vina_pct` | % de la superficie con viñas |
| `cereal_pct` | % con cereales (trigo, avena) |
| `frutal_pct` | % con frutales |
| `hortaliza_pct` | % con hortalizas |
| `trabajadores` | personas que trabajan en el predio |
| `km_chillan` | distancia a Chillán, en kilómetros |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t2_predios.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 66 filas y 9 columnas.

## 1. Conocer los datos

📘 Cada fila es un **predio** de un socio de la cooperativa: su superficie (`hectareas`), qué porcentaje de esa
superficie dedica a viñas, cereales, frutales y hortalizas, cuántas personas trabajan en él y a cuántos kilómetros
de Chillán está. **No hay una columna con «el tipo de predio»**: eso es justamente lo que queremos descubrir.

In [ ]:
datos.head()

In [ ]:
print("Predios:", len(datos), "· Comunas:", datos["comuna"].nunique())
# ✅ Debe mostrar: Predios: 66 · Comunas: 15

## 2. Poner las columnas en la misma escala

📘 El clustering agrupa por **distancia**: dos predios están cerca si sus números se parecen. Pero las hectáreas
van de 1 a más de 150 y los porcentajes de 0 a 100: sin escalar, la columna de números más grandes **mandaría** sobre
las demás. `StandardScaler()` transforma cada columna para que tenga **promedio 0 y desviación 1**: así todas
pesan lo mismo.

In [ ]:
COLUMNAS = ["hectareas", "vina_pct", "cereal_pct", "frutal_pct", "hortaliza_pct", "trabajadores", "km_chillan"]
X = StandardScaler().fit_transform(datos[COLUMNAS])
print("Antes de escalar, hectáreas va de", datos["hectareas"].min(), "a", datos["hectareas"].max())
print("Después de escalar, ¿todas las columnas tienen promedio 0 y desviación 1?",
      bool(np.allclose(X.mean(axis=0), 0) and np.allclose(X.std(axis=0), 1)))
# ✅ Debe mostrar: Antes de escalar, hectáreas va de 1.8 a 187.8  /  Después de escalar, ¿todas las columnas tienen promedio 0 y desviación 1? True

## 3. El dendrograma

📘 El clustering **jerárquico aglomerativo** parte con **cada predio como su propio grupo** y, paso a paso, **une
los dos grupos más parecidos**, hasta que queda uno solo. El **dendrograma** es el árbol que registra esas uniones:
cada predio es una hoja abajo, y la **altura** a la que se unen dos ramas indica **qué tan distintos** eran los
grupos que se unieron.

`linkage(X, "ward")` hace todas las uniones. El método de **Ward** une, en cada paso, los dos grupos que menos
desordenan el resultado (los que, al unirse, quedan más compactos).

In [ ]:
Z = linkage(X, "ward")
plt.figure(figsize=(11, 4.5))
dendrogram(Z, labels=datos["predio"].tolist(), leaf_font_size=7)
plt.title("Dendrograma de los predios (método de Ward)")
plt.ylabel("Altura de la unión (qué tan distintos eran)")
plt.show()

## 4. ¿Dónde cortar el árbol?

📘 Cortar el dendrograma con una línea horizontal decide **cuántos grupos** quedan. La regla práctica es cortar
donde está el **salto más grande**: si para pasar de 4 a 3 grupos hay que unir dos grupos **muy** distintos
(una unión muy alta), conviene quedarse en 4. La celda muestra la altura de las últimas uniones.

In [ ]:
for k in range(7, 1, -1):
    print(f"De {k} a {k - 1} grupos: altura {Z[-(k - 1), 2]:.1f}")
# ✅ Debe mostrar: De 7 a 6 grupos: altura 3.2  /  De 6 a 5 grupos: altura 3.8  /  De 5 a 4 grupos: altura 4.5  /  De 4 a 3 grupos: altura 15.0  /  De 3 a 2 grupos: altura 16.6  /  De 2 a 1 grupos: altura 17.4

### ✍️ ¿Entre qué alturas está el salto más grande? ¿En cuántos grupos conviene cortar el árbol? Marquen ese corte en el dendrograma (una línea horizontal imaginaria) y cuenten las ramas que cruza.

_Su respuesta:_

## 5. Los cuatro grupos y su perfil

📘 `fcluster(Z, 4, "maxclust")` corta el árbol para que queden 4 grupos y le asigna a cada predio el número de su
grupo. Luego se calcula el **perfil** de cada grupo: el promedio de cada columna. El perfil es lo que permite
**ponerle nombre** a cada grupo.

In [ ]:
datos["grupo"] = fcluster(Z, 4, "maxclust")
print("Tamaños de los grupos:", sorted(datos["grupo"].value_counts().tolist()))
# ✅ Debe mostrar: Tamaños de los grupos: [15, 15, 16, 20]

In [ ]:
perfil = datos.groupby("grupo")[COLUMNAS].mean().round(1)
perfil["predios"] = datos["grupo"].value_counts()
perfil

### ✍️ Miren el perfil de cada grupo y pónganle un **nombre** que un dirigente de la cooperativa entienda (por ejemplo, según qué cultiva, su tamaño y dónde está). Justifiquen cada nombre con dos números del perfil.

_Su respuesta:_

## 6. Comparar con k-means

📘 En la Semana 10 el curso agrupó con **k-means**, que necesita saber **de antemano** cuántos grupos buscar.
Aquí se le da k = 4 (lo que sugirió el dendrograma) y se compara con el jerárquico. La tabla cruzada muestra cuántos
predios de cada grupo jerárquico (filas) caen en cada grupo de k-means (columnas). Los números de los grupos son
solo etiquetas: lo que importa es si los predios **quedan juntos**.

In [ ]:
datos["grupo_kmeans"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X).labels_
cruce = pd.crosstab(datos["grupo"], datos["grupo_kmeans"])
cruce

In [ ]:
print("Predios en los que ambos métodos coinciden:", cruce.max(axis=1).sum(), "de", len(datos))
# ✅ Debe mostrar: Predios en los que ambos métodos coinciden: 65 de 66

### ✍️ ¿Coinciden los dos métodos? ¿Qué ventaja tuvo el jerárquico sobre k-means en este caso? ¿Y qué desventaja tendría si la cooperativa tuviera 50.000 predios?

_Su respuesta:_

## 7. ¿Importa cómo se mide la distancia entre grupos?

📘 Para decidir qué dos grupos unir hay que medir la **distancia entre dos grupos**, no solo entre dos predios.
Hay varias formas (se llaman **métodos de enlace**):

| Método | Distancia entre dos grupos |
|---|---|
| `single` (simple) | La de sus dos predios **más cercanos** |
| `complete` (completo) | La de sus dos predios **más lejanos** |
| `average` (promedio) | El promedio de todas las distancias entre sus predios |
| `ward` | Cuánto aumenta el desorden dentro del grupo al unirlos |

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
METODO = "ward"        # ✏️ pruebe con "ward", "complete", "average" y "single" (al terminar, deje "ward")

grupos_prueba = fcluster(linkage(X, METODO), 4, "maxclust")
print(METODO, "→ tamaños de los 4 grupos:", sorted(pd.Series(grupos_prueba).value_counts().tolist()))
plt.figure(figsize=(11, 4))
dendrogram(linkage(X, METODO), labels=datos["predio"].tolist(), leaf_font_size=7)
plt.title(f"Dendrograma con el método {METODO}")
plt.show()

**✏️ Anoten aquí sus resultados:**

| Método | Tamaños de los 4 grupos | ¿Son grupos útiles para la cooperativa? |
|---|:-:|---|
| `ward` | | |
| `complete` | | |
| `average` | | |
| `single` | | |

### ✍️ ¿Qué método da grupos distintos a los demás? Miren su dendrograma: ¿por qué creen que pasa? (Pista: hay unos pocos predios «de transición», que mezclan dos tipos de cultivo.)

_Su respuesta:_

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para la directiva de la Cooperativa Campesina Valle del Itata: ¿cómo usarían los cuatro grupos para organizar la asesoría técnica y los servicios a los socios? Den una acción concreta para al menos tres de los grupos.

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema2.ipynb`, junto con el informe en PDF.